# Raw Netlists and Configurable Xyce Features

**Audience:** Existing Xyce users who already know netlist syntax and want Python orchestration without losing exact simulator control.

This notebook shows two paths: exact raw netlist projects and graph-compiled projects that use configurable feature specs for advanced directives.

In [ ]:
from pathlib import Path
import shutil
from tempfile import TemporaryDirectory

from xyce_py import (
    CircuitGraph,
    OutputSpec,
    Resistor,
    VoltageSource,
    XyceAnalysisSpec,
    XyceFeatureConfig,
    XyceOutputSpec,
    XyceProject,
    find_xyce_executable,
)


def xyce_available() -> bool:
    default_path = Path("/usr/local/XyceNF_7.10/bin/Xyce")
    return default_path.exists() or shutil.which("Xyce") is not None

## Exact raw netlist project

`XyceProject` keeps the netlist text exact. Xyce remains the parser for all simulator semantics.

In [ ]:
raw_project = XyceProject(
    "raw-divider",
    """* raw voltage divider
V1 1 0 DC 10
R1 1 2 1k
R2 2 0 1k
.OP
.PRINT DC FORMAT=CSV FILE=raw.csv V(1) V(2)
.END
""",
    output_specs=(OutputSpec.csv("waveforms", "raw.csv"),),
)

print(raw_project.netlist_content)

In [ ]:
if xyce_available():
    with TemporaryDirectory() as tmpdir:
        result = raw_project.run(xyce_path=find_xyce_executable(), base_out_dir=tmpdir)
        print(result.output("waveforms").frame)
else:
    print("Xyce was not found; raw project construction is still demonstrated above.")

## Configurable feature specs

Use specs when topology should come from `CircuitGraph` but advanced Xyce directives should remain exact text.

In [ ]:
graph = CircuitGraph(xyce_path="Xyce")
graph.add_node("gnd", is_ground=True)
graph.add_branch("vin", "gnd", [VoltageSource("supply", 10.0)])
graph.add_branch("vin", "vout", [Resistor("load", 1000)])

body = graph.compile_body()
vout = body.user_to_spice_node["vout"]

features = XyceFeatureConfig(
    analyses=[
        XyceAnalysisSpec(".NOISE", [f"V({vout})", "V_supply", "DEC", "10", "1", "1e6"]),
    ],
    outputs=[
        XyceOutputSpec("noise", "NOISE", ["ONOISE", "INOISE"], "noise.csv"),
    ],
)

noise_project = graph.compile_project(
    "noise-analysis",
    features.directive_lines(),
    output_specs=features.output_specs(),
)

print(noise_project.netlist_content)